In [127]:
import requests
from bs4 import BeautifulSoup
from time import sleep
from urllib.parse import unquote

import pandas as pd
import numpy as np
import re

In [2]:
HEADERS = {"User-Agent": "qangos-scraper/1.0(contact:q@bradtaba.org)"}
API = "https://onepiece.fandom.com/api.php"

character_columns = [
    'Official English Name',
    'Debut',
    'Affiliations',
    'Occupations',
    'Origin',
    'Status',
    'Age',
    'Birthday',
    'Height',
    'Blood Type',
    'Bounty'
]

In [3]:
def fetch_onepiece(page, sleep_time=1):
    params = {
        "action": "parse",
        "page": unquote(page),
        "prop": "text",
        "format": "json"
    }

    try:
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        data = r.json()
        html = data["parse"]["text"]["*"]
        sleep(sleep_time)
    except:
        return False
    else:
        print(f"Page has been successfully fetched! {page}")

    return html


In [4]:
character_page = fetch_onepiece("List_of_Canon_Characters")
soup = BeautifulSoup(character_page, 'html5lib')

char_table = []
for i in soup.tr.find_next_siblings():
    char_data = {
        'name': i.a.text,
        'link': i.a['href'][2:].split('/')[-1]
    }
    char_table.append(char_data)
    
char_table = pd.DataFrame(char_table)

Page has been successfully fetched! List_of_Canon_Characters


In [5]:
status_pattern = r'\(.*?$'
status_regex = re.compile(status_pattern)

name_pattern = r' \(.*?\)'
name_regex = re.compile(name_pattern)

bday_pattern = r'\b[a-z]+\b \d{1,2}'
bday_regex = re.compile(bday_pattern, re.I)

age_pattern = r'\d+'
age_regex = re.compile(age_pattern)

height_pattern = r'\b\d+\s*cm\b'
height_regex = re.compile(height_pattern)

height_m_pattern = r'\b(\d+)\s*m\b'
height_m_regex = re.compile(height_m_pattern)

height_km_pattern = r'\b(\d+)\s*km\b'
height_km_regex = re.compile(height_km_pattern)

hyperlink_pattern = r'\[\d+\]'
hyperlink_regex = re.compile(hyperlink_pattern)

debut_pattern = r'Chapter (\d+)'
debut_regex = re.compile(debut_pattern, re.I)

bounty_edgecase_pattern = r'\(At least (.*?)\)'
bounty_edgecase_regex = re.compile(bounty_edgecase_pattern, re.I)

bounty_edgecase2_pattern = r'\(([0-9,]*)\)'
bounty_edgecase2_regex = re.compile(bounty_edgecase2_pattern, re.I)

In [6]:
armament_page = fetch_onepiece('Haki/Armament_Haki')
armament_soup = BeautifulSoup(armament_page, 'html5lib')

armament_users = []

for i in armament_soup.find_all('small'):
    armament_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Armament_Haki


In [7]:
conq_page = fetch_onepiece('Haki/Supreme_King_Haki')
conq_soup = BeautifulSoup(conq_page, 'html5lib')

conq_users = []

for i in conq_soup.find_all('small'):
    conq_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Supreme_King_Haki


In [8]:
observation_page = fetch_onepiece('Haki/Observation_Haki')
observation_soup = BeautifulSoup(observation_page, 'html5lib')

observation_users = []

for i in observation_soup.find_all('small'):
    observation_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Observation_Haki


In [9]:
def clean_data(luffy_dict):
    if 'Official English Name' in luffy_dict: 
        luffy_dict['Official English Name'] = name_regex.sub('', luffy_dict['Official English Name'].split(';')[0])

    if 'Occupations' in luffy_dict:
        luffy_dict['Occupations'] = luffy_dict['Occupations'].split(';')[0]

    if 'Bounty' in luffy_dict:
        if luffy_dict['Bounty'].lower().startswith('(') and luffy_dict['Bounty'].lower().endswith(')'):
            luffy_dict['Bounty'] = luffy_dict['Bounty'][1:-1]

        if isinstance(luffy_dict['Bounty'], int):
            luffy_dict['Bounty'] = luffy_dict['Bounty']
        elif bounty_edgecase_regex.findall(luffy_dict['Bounty']):
            luffy_dict['Bounty'] = int(''.join(bounty_edgecase_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
        elif (luffy_dict['Bounty'].lower().startswith('at')):
            if bounty_edgecase2_regex.findall(luffy_dict['Bounty']):
                luffy_dict['Bounty'] = int(''.join(bounty_edgecase2_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
            else:
                luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[-1]).split(',')))
        elif (luffy_dict['Bounty'].lower().startswith('unknown')):
            luffy_dict['Bounty'] = 0
        elif luffy_dict['Bounty'].lower().startswith('¥'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'][1:].split(' ')[0]).split(',')))
        elif luffy_dict['Bounty'].lower().startswith('over'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[1]).split(',')))
        elif luffy_dict['Bounty'].lower().startswith('less'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[2]).split(',')))
        elif bounty_edgecase2_regex.findall(luffy_dict['Bounty']):
            luffy_dict['Bounty'] = int(''.join(bounty_edgecase2_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
        else:
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[0]).split(',')))

    if 'Origin' in luffy_dict:
        luffy_dict['Origin'] = ' '.join(luffy_dict['Origin'].split()[:2])

    if 'Birthday' in luffy_dict:
        luffy_dict['Birthday'] = bday_regex.findall(luffy_dict['Birthday'])[0]

    if 'Age' in luffy_dict:
        ages = age_regex.findall(luffy_dict['Age'].split(';')[-1].strip())
        if ages:
            luffy_dict['Age'] = int(ages[0])
        elif luffy_dict['Age'] == 'Multiple Centuries':
            luffy_dict['Age'] = 100
        else:
            luffy_dict['Age'] = 0

    if 'Height' in luffy_dict:
        cm = height_regex.findall(luffy_dict['Height'])
        m = height_m_regex.findall(luffy_dict['Height'])
        km = height_km_regex.findall(luffy_dict['Height'])
        if cm:
            luffy_dict['Height'] = int(cm[-1][:-3])
        elif m:
            luffy_dict['Height'] = int(m[-1])*100
        elif km:
            luffy_dict['Height'] = int(km[-1])*1000
        else:
            luffy_dict['Height'] = np.nan

    if 'Debut' in luffy_dict:
        chap = debut_regex.findall(luffy_dict['Debut'])
        if chap:
            luffy_dict['Debut'] = int(chap[0])
        else:
            luffy_dict['Debut'] = np.nan

    return luffy_dict

In [10]:
character_info_list = []

In [13]:
for link in char_table['link'].to_list():
    luffy_page = fetch_onepiece(link)
    if not luffy_page:
        print(f'skip: {link}. index {int(char_table[char_table['link']==link].index[0])}')
        print('Next:', int(char_table[char_table['link']==link].index[0])+1)
        continue
    luffy_soup = BeautifulSoup(luffy_page, 'html5lib')

    luffy_dict = {}
    for i in luffy_soup.find_all(class_='pi-data-label pi-secondary-font'):
        if i.text[:-1] in character_columns:
            if i.text[:-1]=='Affiliations':
                luffy_dict[i.text[:-1]] = [a['title'] for a in i.find_next_sibling().find_all('a', title=True)]
            else:
                luffy_dict[i.text[:-1]] = hyperlink_regex.sub(' ', i.find_next_sibling().text).strip()

    fruit_data = luffy_soup.find_all('span', class_='pi-data-label pi-secondary-font')
    if fruit_data:
        luffy_dict['Devil Fruit'] = 1
        luffy_dict['Fruit Type'] = hyperlink_regex.sub(' ', fruit_data[-1].find_next_sibling().text).strip()
    else:
        luffy_dict['Devil Fruit'] = 0

    if 'Official English Name' in luffy_dict:
        luffy_dict['Observation Haki'] = 1 if luffy_dict['Official English Name'] in observation_users else 0
        luffy_dict['Armament Haki'] = 1 if luffy_dict['Official English Name'] in armament_users else 0
        luffy_dict['Conquerers Haki'] = 1 if luffy_dict['Official English Name'] in conq_users else 0
    else:
        luffy_dict['Observation Haki'] = 0
        luffy_dict['Armament Haki'] = 0
        luffy_dict['Conquerers Haki'] = 0

    character_info_list.append(clean_data(luffy_dict))
    print('Next:', int(char_table[char_table['link'] == link].index[0])+1)

Page has been successfully fetched! A_O
Next: 1
Page has been successfully fetched! Abdullah
Next: 2
Page has been successfully fetched! Absalom
Next: 3
Page has been successfully fetched! Acilia
Next: 4
Page has been successfully fetched! Adele
Next: 5
Page has been successfully fetched! Aegir
Next: 6
Page has been successfully fetched! Ageha_Woman
Next: 7
Page has been successfully fetched! Aggie_68
Next: 8
Page has been successfully fetched! Agotogi
Next: 9
Page has been successfully fetched! Agsilly
Next: 10
Page has been successfully fetched! Agyo
Next: 11
Page has been successfully fetched! Ahho_Desunen_IX
Next: 12
Page has been successfully fetched! Ahho_Zurako
Next: 13
Page has been successfully fetched! Ahiru
Next: 14
Page has been successfully fetched! Aisa
Next: 15
Page has been successfully fetched! Akehende
Next: 16
Page has been successfully fetched! Akudai_Kanzaburo
Next: 17
Page has been successfully fetched! Akumai
Next: 18
Page has been successfully fetched! Aladine
N

In [14]:
character_info = pd.DataFrame(character_info_list)
character_info.head()

character_info.to_csv('character_info_v2.csv')

In [48]:
len(character_info)

1555

# Chapter Scraping

In [51]:
cover_pattern = r'\(cover\)'
cover_regex = re.compile(cover_pattern)

flashback_pattern = r'\(flashback\)'
flashback_regex = re.compile(flashback_pattern)

recent_chapter = 1176

In [52]:
chapter_character_dataframe = char_table[['name']].copy(deep=True).set_index('name')

for i in range(1,recent_chapter+1):
    chapter_character_dataframe[str(i)] = 0

/tmp/ipykernel_11354/1725584858.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chapter_character_dataframe[str(i)] = 0
/tmp/ipykernel_11354/1725584858.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chapter_character_dataframe[str(i)] = 0
/tmp/ipykernel_11354/1725584858.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, u

In [53]:
chapter_character_dataframe

,1,2,3,4,5,6,7,8,9,10,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
name,,,,,,,,,,,,,,,,,,,,,
A O,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Abdullah,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Absalom,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Acilia,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adele,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zodia,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Zotto,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Zucca,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [116]:
character_scraped_df = pd.read_csv('character_info_v2.csv')

In [55]:
import ast
import itertools

In [56]:
affiliation_pattern = r'\(.*?\)'
affiliation_regex = re.compile(affiliation_pattern)

weird_pattern = r'\xa0‡'
weird_regex = re.compile(weird_pattern)

In [57]:
affiliations = [i.strip() for i in list(itertools.chain.from_iterable([ast.literal_eval(i) for i in character_scraped_df['Affiliations'].dropna().to_list()]))]
stripped_affiliations = []
for i in affiliations:
    stripped_affiliations.append(affiliation_regex.sub('', i).strip()) 

affiliations_list = list(set(stripped_affiliations))

In [58]:
chapter_affiliations_dataframe = pd.DataFrame(affiliations_list, columns=['Affiliation']).set_index('Affiliation')

for i in range(1,recent_chapter+1):
    chapter_affiliations_dataframe[str(i)] = 0

/tmp/ipykernel_11354/3553183193.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chapter_affiliations_dataframe[str(i)] = 0
/tmp/ipykernel_11354/3553183193.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chapter_affiliations_dataframe[str(i)] = 0
/tmp/ipykernel_11354/3553183193.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fr

In [59]:
chapter_affiliations_dataframe

,1,2,3,4,5,6,7,8,9,10,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
Affiliation,,,,,,,,,,,,,,,,,,,,,
Caesar Clown,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Wapol,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pacifista,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Laboon,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Guardians,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
G-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Tontatta Pirates,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Gol D. Roger,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [83]:
for chapter in range(1,recent_chapter+1):
    chapter_page = fetch_onepiece(f"Chapter_{chapter}")
    chapter_soup = BeautifulSoup(chapter_page, 'html5lib')

    chapter_characters = [i.text for i in chapter_soup.find(class_='CharTable').find_all('li')]
    present_characters = []
    for i in chapter_characters:
        if not (cover_regex.findall(i) or flashback_regex.findall(i)):
            present_characters.append(i)
    for i in present_characters:
        if i in char_table['name'].to_list():
            chapter_character_dataframe.at[i, str(chapter)] = 1

    # chapter_affiliations = [i.text for i in chapter_soup.find(class_='CharTable').find_all('dt')]
    # present_affiliations = []
    # for i in chapter_affiliations:
    #     if not (cover_regex.findall(i) or flashback_regex.findall(i)):
    #         present_affiliations.append(i)
    # for i in present_affiliations:
    #     if i in affiliations_list:
    #         chapter_affiliations_dataframe.at[i, str(chapter)] = 1

Page has been successfully fetched! Chapter_1
Page has been successfully fetched! Chapter_2
Page has been successfully fetched! Chapter_3
Page has been successfully fetched! Chapter_4
Page has been successfully fetched! Chapter_5
Page has been successfully fetched! Chapter_6
Page has been successfully fetched! Chapter_7
Page has been successfully fetched! Chapter_8
Page has been successfully fetched! Chapter_9
Page has been successfully fetched! Chapter_10
Page has been successfully fetched! Chapter_11
Page has been successfully fetched! Chapter_12
Page has been successfully fetched! Chapter_13
Page has been successfully fetched! Chapter_14
Page has been successfully fetched! Chapter_15
Page has been successfully fetched! Chapter_16
Page has been successfully fetched! Chapter_17
Page has been successfully fetched! Chapter_18
Page has been successfully fetched! Chapter_19
Page has been successfully fetched! Chapter_20
Page has been successfully fetched! Chapter_21
Page has been successf

In [84]:
chapter_affiliations_dataframe[chapter_affiliations_dataframe['560']==1]

,1,2,3,4,5,6,7,8,9,10,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
Affiliation,,,,,,,,,,,,,,,,,,,,,
Thriller Bark Pirates,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Kuja Pirates,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Buggy and Alvida Alliance,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Whitebeard Pirates,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
Straw Hat Pirates,0,0,0,0,1,1,1,1,1,1,...,0,0,0,1,1,1,1,1,1,1


In [89]:
chapter_character_dataframe.to_csv('character_appearance.csv')

In [88]:
chapter_character_dataframe[chapter_character_dataframe['562']==1]

,1,2,3,4,5,6,7,8,9,10,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
name,,,,,,,,,,,,,,,,,,,,,
Bartholomew Kuma,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Blenheim,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Boa Hancock,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
Buggy,0,0,0,0,0,0,0,0,1,1,...,1,0,0,0,0,0,0,0,0,0
Crocodile,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Curiel,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Dracule Mihawk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Edward Newgate,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
Emporio Ivankov,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [81]:
"Monkey D. Luffy" in char_table['name'].to_list()

True

# Character Affiliations

In [117]:
char_affil_df = character_scraped_df[['Official English Name']].copy(deep=True).set_index('Official English Name')

for i in affiliations_list:
    char_affil_df[i] = 0

/tmp/ipykernel_11354/3304305546.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  char_affil_df[i] = 0
/tmp/ipykernel_11354/3304305546.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  char_affil_df[i] = 0
/tmp/ipykernel_11354/3304305546.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  char_af

In [118]:
char_affil_df

,Caesar Clown,Wapol,Pacifista,Laboon,Guardians,Lulusia Kingdom,Harahettania,Drake Pirates,Bourgeois Kingdom,Portgas D. Ace,...,Ganzack Pirates,Ohara,MADS,Blackbeard Pirates,G-2,G-1,Tontatta Pirates,Gol D. Roger,Will of D.,Sea Floor
Official English Name,,,,,,,,,,,,,,,,,,,,,
A.O.,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Abdullah,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Absalom,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Agilia,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adelle,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zodia,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Burr,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Zukka,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [137]:
for i in range(len(character_scraped_df)):
    affil = character_scraped_df['Affiliations'][i]
    if affil is np.nan:
        continue
    for j in ast.literal_eval(character_scraped_df['Affiliations'][i]):
        if j in affiliations_list:
            char_affil_df.at[character_scraped_df['Official English Name'][i], j] = 1

In [141]:
char_affil_df.to_csv('character_affiliations.csv')

In [ ]:
char_app_df = pd.read_csv('character_appearance.csv')